# CardioIA — Fase 4 | Parte 1: Pré-processamento e Organização das Imagens

**Assistente Cardiológico Virtual com Visão Computacional**

Este notebook implementa o pipeline completo de preparação de imagens médicas para
classificação com Redes Neurais Convolucionais (CNNs).

## Dataset escolhido

**Chest X-Ray Images (Pneumonia)** — Kaggle ([paultimothymooney/chest-xray-pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia))

Justificativas da escolha:

- **Relevância clínica:** radiografias de tórax são exames diretamente relacionados ao
  contexto cardiopulmonar do projeto CardioIA;
- **Tamanho viável (~2 GB):** permite treino completo no Google Colab gratuito, ao
  contrário do NIH Chest X-rays completo (~42 GB);
- **Problema binário bem definido:** `NORMAL` vs `PNEUMONIA`, ideal para comparar uma
  CNN do zero com Transfer Learning;
- **Desbalanceamento real (~3:1):** reflete um desafio comum em dados de saúde e
  alimenta a discussão de ética/fairness (Ir Além 1).

> ⚙️ **Como executar:** Google Colab → `Ambiente de execução > Alterar tipo de ambiente > GPU (T4)`

## 1. Setup e imports

In [ ]:
import os
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices("GPU"))

## 2. Download do dataset

Usamos a biblioteca `kagglehub`, que baixa e cacheia o dataset sem precisar de
credenciais para datasets públicos.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
DATA_DIR = Path(dataset_path) / "chest_xray"
print("Dataset em:", DATA_DIR)
print("Conteúdo:", sorted(os.listdir(DATA_DIR)))

## 3. Exploração inicial

Antes de qualquer transformação, precisamos entender **o que temos**: quantas imagens
por classe, em que formato e com que dimensões. Isso orienta as decisões de
pré-processamento.

In [ ]:
def contar_imagens(base_dir: Path) -> dict:
    """Conta imagens por split e por classe."""
    contagem = {}
    for split in ["train", "val", "test"]:
        for classe in ["NORMAL", "PNEUMONIA"]:
            pasta = base_dir / split / classe
            n = len(list(pasta.glob("*.jpeg"))) if pasta.exists() else 0
            contagem[(split, classe)] = n
    return contagem


contagem = contar_imagens(DATA_DIR)
for (split, classe), n in contagem.items():
    print(f"{split:>5} | {classe:<9} | {n:>5} imagens")

total = sum(contagem.values())
print(f"\nTotal: {total} imagens")

In [ ]:
# Distribuição de classes — visualização do desbalanceamento
fig, ax = plt.subplots(figsize=(8, 4))
splits = ["train", "val", "test"]
normais = [contagem[(s, "NORMAL")] for s in splits]
pneumonias = [contagem[(s, "PNEUMONIA")] for s in splits]

x = np.arange(len(splits))
ax.bar(x - 0.2, normais, 0.4, label="NORMAL", color="#4caf50")
ax.bar(x + 0.2, pneumonias, 0.4, label="PNEUMONIA", color="#e53935")
ax.set_xticks(x, splits)
ax.set_ylabel("Nº de imagens")
ax.set_title("Distribuição de classes por conjunto — dataset original")
ax.legend()
plt.tight_layout()
plt.show()

ratio = contagem[("train", "PNEUMONIA")] / contagem[("train", "NORMAL")]
print(f"Desbalanceamento no treino: {ratio:.2f} PNEUMONIA para cada NORMAL")

**Observações importantes:**

1. O conjunto `val/` original tem só **16 imagens** — insuficiente para validação
   confiável. Vamos recriar a validação a partir do treino (80/20).
2. Há **desbalanceamento ~3:1** a favor de PNEUMONIA — por isso a acurácia sozinha
   engana; usaremos também precision, recall e F1 na Parte 2, e `class_weight` no treino.

In [ ]:
# Inspeção de amostras: dimensões e modos de cor variam entre as imagens
amostras = list((DATA_DIR / "train" / "NORMAL").glob("*.jpeg"))[:5] + \
           list((DATA_DIR / "train" / "PNEUMONIA").glob("*.jpeg"))[:5]

for p in amostras[:6]:
    with Image.open(p) as img:
        print(f"{p.parent.name:<9} | {img.size} | modo={img.mode}")

In [ ]:
# Visualização de exemplos de cada classe
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, classe in zip(axes, ["NORMAL", "PNEUMONIA"]):
    imgs = sorted((DATA_DIR / "train" / classe).glob("*.jpeg"))[:4]
    for a, p in zip(ax, imgs):
        a.imshow(Image.open(p).convert("L"), cmap="gray")
        a.set_title(classe, fontsize=10)
        a.axis("off")
plt.suptitle("Amostras do dataset (raio-X de tórax)")
plt.tight_layout()
plt.show()

## 4. Decisões de pré-processamento

| Etapa | Decisão | Justificativa |
|---|---|---|
| **Redimensionamento** | 150×150 px | CNNs exigem entrada de tamanho fixo; 150px preserva padrões pulmonares visíveis e cabe na memória da GPU gratuita do Colab |
| **Conversão de formato** | JPEG → tensores RGB (3 canais) | Embora raios-X sejam em escala de cinza, modelos pré-treinados (VGG16/ResNet) esperam 3 canais — manter RGB permite reutilizar o mesmo pipeline na Parte 2 |
| **Normalização** | Pixels [0, 255] → [0, 1] | Gradientes mais estáveis e convergência mais rápida no treino |
| **Divisão** | treino 80% / validação 20% (do train original) + teste original intacto | O `val/` original (16 imgs) é estatisticamente inútil; o teste nunca é tocado para garantir avaliação honesta |
| **Data augmentation** | rotação ±10%, zoom ±10%, contraste ±10% — **somente no treino** | Aumenta a diversidade efetiva dos dados e reduz overfitting; flip horizontal foi **evitado** pois inverteria a posição anatômica do coração |

In [ ]:
# Constantes do pipeline (reutilizadas na Parte 2)
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
VAL_SPLIT = 0.20

## 5. Criação dos conjuntos de treino, validação e teste

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,  # ordem fixa para gerar matriz de confusão depois
)

class_names = train_ds.class_names
print("Classes:", class_names)  # ['NORMAL', 'PNEUMONIA'] -> 0 = NORMAL, 1 = PNEUMONIA

## 6. Normalização e data augmentation

A normalização é aplicada a **todos** os conjuntos. O augmentation é aplicado
**apenas ao treino** — validação e teste devem refletir a distribuição real dos dados.

In [ ]:
normalization = tf.keras.layers.Rescaling(1.0 / 255)

data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(0.10),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10),
    ],
    name="data_augmentation",
)

# Pipeline tf.data com cache e prefetch (desempenho de leitura)
AUTOTUNE = tf.data.AUTOTUNE


def preparar(ds, augment=False, shuffle=False):
    ds = ds.map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )
    return ds.prefetch(AUTOTUNE)


train_prep = preparar(train_ds, augment=True, shuffle=True)
val_prep = preparar(val_ds)
test_prep = preparar(test_ds)

In [ ]:
# Verificação: faixa de valores após a normalização
batch_imgs, batch_labels = next(iter(train_prep))
print("Shape do batch:", batch_imgs.shape)
print(f"Faixa de pixels: [{batch_imgs.numpy().min():.3f}, {batch_imgs.numpy().max():.3f}]")

In [ ]:
# Visualização do efeito do augmentation sobre uma mesma imagem
img_original = next(iter(train_ds.take(1)))[0][0]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(img_original.numpy().astype("uint8"))
axes[0].set_title("Original")
axes[0].axis("off")
for i, ax in enumerate(axes[1:]):
    aug = data_augmentation(tf.expand_dims(img_original, 0), training=True)[0]
    ax.imshow(aug.numpy().astype("uint8"))
    ax.set_title(f"Augmentada {i + 1}")
    ax.axis("off")
plt.suptitle("Efeito do data augmentation (aplicado somente no treino)")
plt.tight_layout()
plt.show()

## 7. Pesos de classe (mitigação do desbalanceamento)

Como há ~3x mais PNEUMONIA que NORMAL no treino, calculamos `class_weight` para
que erros na classe minoritária pesem mais na função de perda. Esses pesos serão
usados no treino da Parte 2.

In [ ]:
n_normal = contagem[("train", "NORMAL")]
n_pneu = contagem[("train", "PNEUMONIA")]
n_total = n_normal + n_pneu

class_weight = {
    0: n_total / (2.0 * n_normal),    # NORMAL (minoritária -> peso maior)
    1: n_total / (2.0 * n_pneu),      # PNEUMONIA
}
print("Pesos de classe:", class_weight)

## 8. Resumo do pipeline

```
Imagens JPEG (tamanhos variados, ~3:1 desbalanceadas)
       │
       ├── Redimensionamento → 150×150×3
       ├── Normalização      → pixels em [0, 1]
       ├── Split             → treino 80% │ validação 20% │ teste (intacto)
       ├── Augmentation      → rotação/zoom/contraste (apenas treino)
       └── class_weight      → compensação do desbalanceamento
       │
       ▼
Conjuntos prontos para a CNN (Parte 2)
```

**Próximo passo:** o notebook `02_cnn_transfer_learning.ipynb` consome exatamente
este pipeline para treinar a CNN do zero e o modelo com Transfer Learning (VGG16).